# Image Processing and Segmentation

This notebook segments the MRI-derived inputs needed for the brain mesh generation part of the project. It takes the subject's structural MRI and anatomical segmentation, performs image registration and tissue-processing steps, and combines the outputs into a cleaned geometry volume that can be used downstream for mesh creation. We perform the following steps: 
- aligns the subject images to a standard orientation,
- uses FSL tools to extract tissue and surface information,
- combines FreeSurfer- and FSL-derived masks,
- and produces a final `pre_model.nii.gz` volume for the later mesh-generation workflow.

## Before running
Please make sure:
1. FSL is installed and activated in the current environment.
2. The required supporting files are available in `src/dependencies/`.
3. You are running this notebook from the repository's `notebooks/` or `docs/` folder.
4. The subject input files are present in `data/subjects/<subject_name>/img/fs_seg/`.

## Path configuration

In [1]:
# Automatically print the runtime of each executed cell
%load_ext autotime
%matplotlib inline

time: 1.92 s (started: 2026-04-28 12:40:41 +01:00)


In [2]:
from pathlib import Path
import os

# --------------------------------------------------
# [1/5] User settings
# --------------------------------------------------
subject_name = "sub0045"
output_name = "output"

threshold_filter = 10
skull_privacy_percentage = 10

T1 = "T1.nii.gz"

# --------------------------------------------------
# [2/5] Project paths
# --------------------------------------------------
# Assumes the notebook is run from the repository's notebooks/ folder.
PROJECT_ROOT = Path.cwd().resolve().parent

SRC_DIR = PROJECT_ROOT / "src"
DEPENDENCIES_DIR = SRC_DIR / "dependencies"
SUBJECTS_DIR = PROJECT_ROOT / "data" / "subjects"

SUBJECT_DIR = SUBJECTS_DIR / subject_name
IMG_DIR = SUBJECT_DIR / "img"
FS_SEG_DIR = IMG_DIR / "fs_seg"
TMP_DIR = SUBJECT_DIR / "tmp"
FAST_DIR = IMG_DIR / "fast"
BET_DIR = IMG_DIR / "bet"
OUTPUT_DIR = SUBJECT_DIR / output_name

# --------------------------------------------------
# [3/5] Create working folders
# --------------------------------------------------
TMP_DIR.mkdir(parents=True, exist_ok=True)
FAST_DIR.mkdir(parents=True, exist_ok=True)
BET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# [4/5] Export variables for later bash cells
# --------------------------------------------------
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["SRC_DIR"] = str(SRC_DIR)
os.environ["DEPENDENCIES_DIR"] = str(DEPENDENCIES_DIR)
os.environ["SUBJECTS_DIR"] = str(SUBJECTS_DIR)

os.environ["SUBJECT_DIR"] = str(SUBJECT_DIR)
os.environ["IMG_DIR"] = str(IMG_DIR)
os.environ["FS_SEG_DIR"] = str(FS_SEG_DIR)
os.environ["TMP_DIR"] = str(TMP_DIR)
os.environ["FAST_DIR"] = str(FAST_DIR)
os.environ["BET_DIR"] = str(BET_DIR)
os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)

os.environ["SUBJECT_NAME"] = subject_name
os.environ["OUTPUT_NAME"] = output_name
os.environ["T1"] = T1

os.environ["THRESHOLD_FILTER"] = str(threshold_filter)
os.environ["SKULL_PRIVACY_PERCENTAGE"] = str(skull_privacy_percentage)

# --------------------------------------------------
# [5/5] Short setup summary
# --------------------------------------------------
brain_file = FS_SEG_DIR / "brain.nii.gz"
t1_file = FS_SEG_DIR / "T1.nii.gz"
aseg_file = FS_SEG_DIR / "aseg.nii.gz"

print("Setup complete.")
print(f"Subject: {subject_name}")
print(f"Input files: {brain_file.name}, {t1_file.name}, {aseg_file.name}")
print(f"Working folders: {TMP_DIR.name}/, {FAST_DIR.name}/, {BET_DIR.name}/, {OUTPUT_DIR.name}/")

Setup complete.
Subject: sub0045
Input files: brain.nii.gz, T1.nii.gz, aseg.nii.gz
Working folders: tmp/, fast/, bet/, output/
time: 5.03 ms (started: 2026-04-28 12:40:46 +01:00)


## Image reorientation and registration

First, the subject images are reoriented and aligned to a standard MNI reference space using a rigid registration.

This step brings the images into a consistent orientation without changing the subject's overall anatomy. The transform is estimated from the skull-stripped brain image, then applied to the T1 image and the anatomical segmentation so that all inputs remain aligned with each other.

In [3]:
%%bash
set -euo pipefail

# Standard-space reference image
MNI_REF="${DEPENDENCIES_DIR}/MNI152_T1_1mm_brain.nii.gz"

# input_brain="$TMP_DIR/brain_reorient.nii.gz"
# input_t1="$TMP_DIR/T1_reorient.nii.gz"
# input_aseg="$TMP_DIR/aseg_reorient.nii.gz"


input_brain="$FS_SEG_DIR/brain.nii.gz"
input_t1="$FS_SEG_DIR/T1.nii.gz"
input_aseg="$FS_SEG_DIR/aseg.nii.gz"

output_brain="$FAST_DIR/brain_std.nii.gz"
output_t1="$TMP_DIR/T1_std.nii.gz"
output_aseg="$TMP_DIR/aseg_std.nii.gz"
transform_mat="$TMP_DIR/brain_to_mni.mat"

echo "=== Image registration ==="
echo "Subject: $SUBJECT_NAME"
echo "Reference: ${MNI_REF##*/}"
echo

# --------------------------------------------------
# [1/3] Register brain to MNI space
# --------------------------------------------------
echo "[1/3] Registering input: ${input_brain##*/}"
echo "      Output image: ${output_brain##*/}"
echo "      Output transform: ${transform_mat##*/}"

flirt -in "$input_brain" \
      -ref "$MNI_REF" \
      -out "$output_brain" \
      -omat "$transform_mat" \
      -dof 6 \
      -searchrx -90 90 \
      -searchry -90 90 \
      -searchrz -90 90

[ -f "$output_brain" ] && echo "      Status: created"
[ -f "$transform_mat" ] && echo "      Status: saved"
echo

# --------------------------------------------------
# [2/3] Apply the same transform to T1
# --------------------------------------------------
echo "[2/3] Transforming input: ${input_t1##*/}"
echo "      Output image: ${output_t1##*/}"

flirt -in "$input_t1" \
      -ref "$MNI_REF" \
      -out "$output_t1" \
      -applyxfm \
      -init "$transform_mat" \
      -interp trilinear

[ -f "$output_t1" ] && echo "      Status: created"
echo

# --------------------------------------------------
# [3/3] Apply the same transform to aseg
# --------------------------------------------------
echo "[3/3] Transforming input: ${input_aseg##*/}"
echo "      Output image: ${output_aseg##*/}"
echo "      Interpolation: nearest neighbour"

flirt -in "$input_aseg" \
      -ref "$MNI_REF" \
      -out "$output_aseg" \
      -applyxfm \
      -init "$transform_mat" \
      -interp nearestneighbour

[ -f "$output_aseg" ] && echo "      Status: created"
echo
echo "Image registration complete."

=== Image registration ===
Subject: sub0045
Reference: MNI152_T1_1mm_brain.nii.gz

[1/3] Registering input: brain.nii.gz
      Output image: brain_std.nii.gz
      Output transform: brain_to_mni.mat
      Status: created
      Status: saved

[2/3] Transforming input: T1.nii.gz
      Output image: T1_std.nii.gz
      Status: created

[3/3] Transforming input: aseg.nii.gz
      Output image: aseg_std.nii.gz
      Interpolation: nearest neighbour
      Status: created

Image registration complete.
time: 26.6 s (started: 2026-04-28 12:40:56 +01:00)


## Image segmentation

Next, the workflow combines information from FreeSurfer and FSL.

FreeSurfer provides detailed anatomical labels, while FSL is used here to identify broader tissue classes such as grey matter, white matter, and CSF. Using both together gives a more complete starting point for the later geometry and mesh-generation steps.

### Running _fast_

`FAST` segments the registered brain image into grey matter, white matter, and CSF.

In this workflow, these tissue masks are mainly used to improve the combined segmentation volume, especially for CSF and tissue regions that may be incomplete in the original anatomical label map.

In [4]:
%%bash
set -euo pipefail

input_t1="$TMP_DIR/T1_std.nii.gz"
input_aseg="$TMP_DIR/aseg_std.nii.gz"
check_mat="$TMP_DIR/T1_to_aseg.mat"
fast_input="$FAST_DIR/brain_std.nii.gz"

echo "=== FAST tissue segmentation ==="
echo "Subject: $SUBJECT_NAME"
echo

# --------------------------------------------------
# [1/2] Optional alignment check
# --------------------------------------------------
echo "[1/2] Running a quick alignment check"
echo "      Input image: ${input_t1##*/}"
echo "      Reference image: ${input_aseg##*/}"
echo "      Output transform: ${check_mat##*/}"

flirt -in "$input_t1" \
      -ref "$input_aseg" \
      -omat "$check_mat" \
      -nosearch \
      -dof 6

[ -f "$check_mat" ] && echo "      Status: saved"
echo

# --------------------------------------------------
# [2/2] Run FAST on the registered brain image
# --------------------------------------------------
echo "[2/2] Segmenting input: ${fast_input##*/}"
echo "      Output prefix: brain_std_*"

cd "$FAST_DIR"
fast "${fast_input##*/}"

[ -f "$FAST_DIR/brain_std_seg.nii.gz" ] && echo "      Status: segmentation created"
echo
echo "FAST segmentation complete."

=== FAST tissue segmentation ===
Subject: sub0045

[1/2] Running a quick alignment check
      Input image: T1_std.nii.gz
      Reference image: aseg_std.nii.gz
      Output transform: T1_to_aseg.mat
      Status: saved

[2/2] Segmenting input: brain_std.nii.gz
      Output prefix: brain_std_*
      Status: segmentation created

FAST segmentation complete.
time: 3min 3s (started: 2026-04-28 12:42:15 +01:00)


### Running _bet_

`BET` and `BETSURF` are used to estimate surfaces around the head.

These outputs help define the outer skull and skin boundaries, which are later added to the combined volume used for mesh generation.

In [5]:
%%bash
set -euo pipefail

input_t1="$TMP_DIR/T1_std"
bet_output_prefix="$BET_DIR/T1_bet"
identity_mat="$BET_DIR/identity.mat"
betsurf_mesh="$BET_DIR/T1_bet_mesh.vtk"
betsurf_prefix="$BET_DIR/betsurf"

echo "=== BET and BETSURF ==="
echo "Subject: $SUBJECT_NAME"
echo

# --------------------------------------------------
# [1/3] Run BET on the registered T1 image
# --------------------------------------------------
echo "[1/3] Running BET on input: ${input_t1##*/}"
echo "      Output prefix: ${bet_output_prefix##*/}"

bet "$input_t1" "$bet_output_prefix" -s -f 0.35 -g -0.05 -e -R

[ -f "${bet_output_prefix}.nii.gz" ] && echo "      Status: BET output created"
echo

# --------------------------------------------------
# [2/3] Create identity transform for BETSURF
# --------------------------------------------------
echo "[2/3] Preparing transform: ${identity_mat##*/}"

if [ ! -f "$identity_mat" ]; then
  printf "1 0 0 0\n0 1 0 0\n0 0 1 0\n0 0 0 1\n" > "$identity_mat"
fi

[ -f "$identity_mat" ] && echo "      Status: ready"
echo

# --------------------------------------------------
# [3/3] Run BETSURF
# --------------------------------------------------
echo "[3/3] Running BETSURF"
echo "      Input image: ${input_t1##*/}"
echo "      Output mesh: ${betsurf_mesh##*/}"
echo "      Output prefix: ${betsurf_prefix##*/}"

betsurf --t1only -m \
        "$input_t1" \
        "$betsurf_mesh" \
        "$identity_mat" \
        "$betsurf_prefix"

[ -f "$BET_DIR/betsurf_outskin_mask.nii.gz" ] && echo "      Status: surface masks created"
echo
echo "BET and BETSURF complete."

=== BET and BETSURF ===
Subject: sub0045

[1/3] Running BET on input: T1_std
      Output prefix: T1_bet
      Status: BET output created

[2/3] Preparing transform: identity.mat
      Status: ready

[3/3] Running BETSURF
      Input image: T1_std
      Output mesh: T1_bet_mesh.vtk
      Output prefix: betsurf
      Status: surface masks created

BET and BETSURF complete.
time: 31.8 s (started: 2026-04-28 12:45:19 +01:00)


In [6]:
%%bash
set -euo pipefail

input_t1="$TMP_DIR/T1_std.nii.gz"
wholehead_mask="$TMP_DIR/wholehead_mask.nii.gz"
outskin_mask="$BET_DIR/betsurf_outskin_mask.nii.gz"

echo "=== Outer skin mask update ==="
echo "Subject: $SUBJECT_NAME"
echo

# --------------------------------------------------
# [1/2] Create a whole-head mask from T1
# --------------------------------------------------
echo "[1/2] Creating whole-head mask"
echo "      Input image: ${input_t1##*/}"
echo "      Output image: ${wholehead_mask##*/}"

fslmaths "$input_t1" \
         -thr 39 -bin \
         "$wholehead_mask"

[ -f "$wholehead_mask" ] && echo "      Status: created"
echo

# --------------------------------------------------
# [2/2] Merge with BETSURF outer skin mask
# --------------------------------------------------
echo "[2/2] Updating output: ${outskin_mask##*/}"
echo "      Inputs: ${wholehead_mask}, ${outskin_mask##*/}"

fslmaths "$wholehead_mask" \
         -add "$outskin_mask" \
         -bin "$outskin_mask"

[ -f "$outskin_mask" ] && echo "      Status: updated"
echo
echo "Outer skin mask update complete."

=== Outer skin mask update ===
Subject: sub0045

[1/2] Creating whole-head mask
      Input image: T1_std.nii.gz
      Output image: wholehead_mask.nii.gz
      Status: created

[2/2] Updating output: betsurf_outskin_mask.nii.gz
      Inputs: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/tmp/wholehead_mask.nii.gz, betsurf_outskin_mask.nii.gz
      Status: updated

Outer skin mask update complete.
time: 1.55 s (started: 2026-04-28 12:45:51 +01:00)


## Creating brain geometry

In the final preprocessing step, the FreeSurfer- and FSL-derived outputs are combined into one volume.

This step adds missing brain regions, improves the CSF layer, adds skull and skin boundaries, and produces a final combined image that is used in the later mesh-generation workflow.

In [7]:
%%bash
set -euo pipefail

PRIVACY="${SKULL_PRIVACY_PERCENTAGE}"

fast_seg="$FAST_DIR/brain_std_seg.nii.gz"
aseg_std="$TMP_DIR/aseg_std.nii.gz"
final_output="$SUBJECT_DIR/pre_model.nii.gz"

echo "=== Creating brain geometry ==="
echo "Subject: $SUBJECT_NAME"
echo

# --------------------------------------------------
# [1/6] Extract FAST tissue classes
# --------------------------------------------------
echo "[1/6] Extracting FAST tissue classes"
echo "      Input image: ${fast_seg##*/}"
echo "      Outputs: csf_fast.nii.gz, gm_fast.nii.gz, wm_fast.nii.gz"

fslmaths "$fast_seg" -thr 0.5 -uthr 1.5 -bin "$TMP_DIR/csf_fast.nii.gz"
fslmaths "$fast_seg" -thr 1.5 -uthr 2.5 -bin "$TMP_DIR/gm_fast.nii.gz"
fslmaths "$fast_seg" -thr 2.5 -uthr 3.5 -bin "$TMP_DIR/wm_fast.nii.gz"

[ -f "$TMP_DIR/csf_fast.nii.gz" ] && echo "      Status: tissue masks created"
echo

# --------------------------------------------------
# [2/6] Add missing brain tissue to aseg
# --------------------------------------------------
echo "[2/6] Updating the anatomical segmentation"
echo "      Input image: ${aseg_std##*/}"
echo "      Output image: aseg_bm.nii.gz"

fslmaths "$TMP_DIR/aseg_std.nii.gz" -bin "$TMP_DIR/aseg_std_bin.nii.gz"
fslmaths "$TMP_DIR/gm_fast.nii.gz" -bin "$TMP_DIR/gm_fast_bin.nii.gz"
fslmaths "$TMP_DIR/wm_fast.nii.gz" -bin "$TMP_DIR/wm_fast_bin.nii.gz"
fslmaths "$TMP_DIR/wm_fast_bin.nii.gz" -add "$TMP_DIR/gm_fast_bin.nii.gz" "$TMP_DIR/brain_fast.nii.gz"
fslmaths "$TMP_DIR/brain_fast.nii.gz" -sub "$TMP_DIR/aseg_std_bin.nii.gz" -thr 1 -ero -dilM -dilM -ero -bin -mul 300 "$TMP_DIR/bm.nii.gz"

fslmaths "$TMP_DIR/bm.nii.gz" -bin -mul 1000 "$TMP_DIR/bm_bin.nii.gz"
fslmaths "$TMP_DIR/aseg_std.nii.gz" -sub "$TMP_DIR/bm_bin.nii.gz" -thr 1 "$TMP_DIR/aseg_std.nii.gz"
fslmaths "$TMP_DIR/bm.nii.gz" -add "$TMP_DIR/aseg_std.nii.gz" "$TMP_DIR/aseg_bm.nii.gz"

[ -f "$TMP_DIR/aseg_bm.nii.gz" ] && echo "      Status: updated"
echo

# --------------------------------------------------
# [3/6] Add CSF outside the ventricles
# --------------------------------------------------
echo "[3/6] Adding the CSF layer"
echo "      Output image: aseg_bm_csf.nii.gz"

fslmaths "$TMP_DIR/csf_fast.nii.gz" -dilM -ero "$TMP_DIR/csf_fast.nii.gz"
fslmaths "$TMP_DIR/aseg_bm.nii.gz" -thr 42.5 -uthr 43.5 "$TMP_DIR/ventricleR.nii.gz"
fslmaths "$TMP_DIR/aseg_bm.nii.gz" -thr 3.5 -uthr 4.5 "$TMP_DIR/ventricleL.nii.gz"
fslmaths "$TMP_DIR/ventricleR.nii.gz" -add "$TMP_DIR/ventricleL.nii.gz" "$TMP_DIR/ventricles_both.nii.gz"
fslmaths "$TMP_DIR/ventricles_both.nii.gz" -bin "$TMP_DIR/ventricles_both.nii.gz"
fslmaths "$TMP_DIR/csf_fast.nii.gz" -sub "$TMP_DIR/ventricles_both.nii.gz" "$TMP_DIR/csf_no_vent.nii.gz"
fslmaths "$TMP_DIR/csf_no_vent.nii.gz" -thr 0.9 -uthr 1.1 "$TMP_DIR/csf_no_vent.nii.gz"

fslmaths "$TMP_DIR/aseg_bm.nii.gz" -bin "$TMP_DIR/aseg_bm_bin.nii.gz"
fslmaths "$TMP_DIR/csf_no_vent.nii.gz" -bin -dilM -ero "$TMP_DIR/csf_no_vent_bin.nii.gz"
fslmaths "$TMP_DIR/csf_no_vent_bin.nii.gz" -sub "$TMP_DIR/aseg_bm_bin.nii.gz" -thr 1 -mul 256 "$TMP_DIR/csf.nii.gz"
fslmaths "$TMP_DIR/aseg_bm.nii.gz" -add "$TMP_DIR/csf.nii.gz" "$TMP_DIR/aseg_bm_csf.nii.gz"

fslmaths "$TMP_DIR/aseg_bm_csf.nii.gz" -bin "$TMP_DIR/aseg_bm_csf_bin.nii.gz"
fslmaths "$TMP_DIR/aseg_bm_csf_bin.nii.gz" -dilM -dilM -ero -ero "$TMP_DIR/aseg_bm_csf_bin_mask.nii.gz"
fslmaths "$TMP_DIR/aseg_bm_csf_bin_mask.nii.gz" -sub "$TMP_DIR/aseg_bm_csf_bin.nii.gz" "$TMP_DIR/csf_missing.nii.gz"
fslmaths "$TMP_DIR/csf_missing.nii.gz" -mul 256 "$TMP_DIR/csf_missing.nii.gz"
fslmaths "$TMP_DIR/csf_missing.nii.gz" -add "$TMP_DIR/aseg_bm_csf.nii.gz" "$TMP_DIR/aseg_bm_csf.nii.gz"

[ -f "$TMP_DIR/aseg_bm_csf.nii.gz" ] && echo "      Status: CSF layer added"
echo

# --------------------------------------------------
# [4/6] Add skull around the brain and CSF
# --------------------------------------------------
echo "[4/6] Adding the skull layer"
echo "      Output image: aseg_bm_csf_skull.nii.gz"

fslmaths "$TMP_DIR/aseg_bm_csf.nii.gz" -bin -dilM -dilM -ero -ero "$TMP_DIR/aseg_bm_csf_bin.nii.gz"
fslmaths "$TMP_DIR/aseg_bm_csf_bin.nii.gz" -ero -dilM -dilM "$TMP_DIR/aseg_bm_csf_bin_dil.nii.gz"
fslmaths "$TMP_DIR/aseg_bm_csf_bin_dil.nii.gz" -sub "$TMP_DIR/aseg_bm_csf_bin.nii.gz" "$TMP_DIR/skull_outline.nii.gz"

fslmaths "$BET_DIR/betsurf_outskull_mask.nii.gz" -sub "$TMP_DIR/aseg_bm_csf_bin.nii.gz" "$TMP_DIR/T1_std_roi.nii.gz"
fslmaths "$TMP_DIR/T1_std_roi.nii.gz" -add "$TMP_DIR/skull_outline.nii.gz" "$TMP_DIR/skull_mask.nii.gz"
fslmaths "$TMP_DIR/skull_mask.nii.gz" -bin "$TMP_DIR/skull_mask.nii.gz"

fslmaths "$TMP_DIR/aseg_bm_csf.nii.gz" -bin "$TMP_DIR/aseg_bm_csf_bin.nii.gz"
fslmaths "$TMP_DIR/skull_mask.nii.gz" -sub "$TMP_DIR/aseg_bm_csf_bin.nii.gz" -thr 1 "$TMP_DIR/skull_mask.nii.gz"
fslmaths "$TMP_DIR/skull_mask.nii.gz" -mul 257 "$TMP_DIR/skull_mask.nii.gz"
fslmaths "$TMP_DIR/aseg_bm_csf.nii.gz" -add "$TMP_DIR/skull_mask.nii.gz" "$TMP_DIR/aseg_bm_csf_skull.nii.gz"

[ -f "$TMP_DIR/aseg_bm_csf_skull.nii.gz" ] && echo "      Status: skull layer added"
echo

# --------------------------------------------------
# [5/6] Add skin and apply privacy cropping
# --------------------------------------------------
echo "[5/6] Adding the skin layer"
echo "      Output image: aseg_bm_csf_skull_skin.nii.gz"

fslmaths "$TMP_DIR/aseg_bm_csf_skull.nii.gz" -bin "$TMP_DIR/aseg_bm_csf_skull_bin.nii.gz"
fslmaths "$BET_DIR/betsurf_outskin_mask.nii.gz" -sub "$TMP_DIR/aseg_bm_csf_skull_bin.nii.gz" "$TMP_DIR/skin_mask.nii.gz"

dim3=$(fslval "$TMP_DIR/T1_std.nii.gz" dim3)
dim3_min=$(( dim3 * PRIVACY / 100 ))

fslmaths "$TMP_DIR/skin_mask.nii.gz" \
         -roi 0 -1 0 -1 "$dim3_min" -1 0 -1 \
         -bin "$TMP_DIR/skin_mask_anon.nii.gz"

fslmaths "$TMP_DIR/skin_mask_anon.nii.gz" -mul 260 "$TMP_DIR/skin_mask_anon.nii.gz"
fslmaths "$TMP_DIR/skin_mask_anon.nii.gz" -add "$TMP_DIR/aseg_bm_csf_skull.nii.gz" "$TMP_DIR/aseg_bm_csf_skull_skin.nii.gz"

[ -f "$TMP_DIR/aseg_bm_csf_skull_skin.nii.gz" ] && echo "      Status: skin layer added"
echo

# --------------------------------------------------
# [6/6] Save the final combined volume
# --------------------------------------------------
echo "[6/6] Saving final output"
echo "      Output file: ${final_output##*/}"

cp "$TMP_DIR/aseg_bm_csf_skull_skin.nii.gz" "$final_output"

[ -f "$final_output" ] && echo "      Status: saved"
echo
echo "Brain geometry creation complete."

=== Creating brain geometry ===
Subject: sub0045

[1/6] Extracting FAST tissue classes
      Input image: brain_std_seg.nii.gz
      Outputs: csf_fast.nii.gz, gm_fast.nii.gz, wm_fast.nii.gz
      Status: tissue masks created

[2/6] Updating the anatomical segmentation
      Input image: aseg_std.nii.gz
      Output image: aseg_bm.nii.gz
      Status: updated

[3/6] Adding the CSF layer
      Output image: aseg_bm_csf.nii.gz
      Status: CSF layer added

[4/6] Adding the skull layer
      Output image: aseg_bm_csf_skull.nii.gz
      Status: skull layer added

[5/6] Adding the skin layer
      Output image: aseg_bm_csf_skull_skin.nii.gz
      Status: skin layer added

[6/6] Saving final output
      Output file: pre_model.nii.gz
      Status: saved

Brain geometry creation complete.
time: 57 s (started: 2026-04-28 12:45:53 +01:00)
